# Block 4 lecture — when one row has several partners

**The lecture's queries, as they were shown: for review after class. Nothing here is typed in class.**

The cell numbers match the slides ("cell 1" to "cell 11"). Run the two setup cells first, then any numbered cell.
Before you run one, say what you expect it to return: that is the habit the lecture practised.

This notebook uses three tables: `orders`, `order_items`, `customers`. The lab points to its cells by number. At the end: the
Block 3 cells the lab cites, copied here; what to remember from Block 4; and the common mistakes.

In [ ]:
import os
from pathlib import Path

import duckdb
import pandas as pd

pd.set_option("display.max_rows", 400)   # up to 400 rows in full; a longer result prints head and tail with "..." between

# Anchor to the project folder (DS1, Block 1), then stand there: every path below is from the project folder.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(PROJECT_ROOT)
print("Working in:", Path.cwd())   # must print this project's folder

con = duckdb.connect()             # an in-memory database; it reads the files in data/raw/ directly

In [ ]:
# Every table as a view with its file's name, so a query can say FROM orders instead of the path.
# A view is a saved query (Lab 1): nothing is copied; the file in data/raw/ is read each time.
TABLES = ["orders", "order_items", "customers"]
for t in TABLES:
    con.sql(f"CREATE OR REPLACE VIEW {t} AS SELECT * FROM 'data/raw/{t}.csv'")
print("views:", ", ".join(TABLES))

### Cell 1 — Orders per customer state (Block 3's cell 3)

In [ ]:
con.sql("""
    SELECT c.customer_state, COUNT(*) AS orders
    FROM orders AS o
    JOIN customers AS c ON c.customer_id = o.customer_id
    GROUP BY c.customer_state
    ORDER BY orders DESC
    LIMIT 5
""").df()

### Cell 2 — The same count, after one more join

In [ ]:
con.sql("""
    SELECT c.customer_state, COUNT(*) AS orders
    FROM orders AS o
    JOIN customers AS c ON c.customer_id = o.customer_id
    JOIN order_items AS i ON i.order_id = o.order_id
    GROUP BY c.customer_state
    ORDER BY orders DESC
    LIMIT 5
""").df()

### Cell 3 — What is one row now? One order's rows in that join

In [ ]:
con.sql("""
    SELECT left(o.order_id, 8) AS order_id, c.customer_state, i.order_item_id, i.price
    FROM orders AS o
    JOIN customers AS c ON c.customer_id = o.customer_id
    JOIN order_items AS i ON i.order_id = o.order_id
    WHERE left(o.order_id, 8) = '181ff95f'
    ORDER BY i.order_item_id
""").df()

### Cell 4 — `HAVING`: which orders have more than one item? One row per order that repeats

In [ ]:
con.sql("""
    SELECT order_id, COUNT(*) AS items
    FROM order_items
    GROUP BY order_id
    HAVING COUNT(*) > 1
    ORDER BY items DESC, order_id
    LIMIT 5
""").df()

### Cell 5 — The same condition in `WHERE`

In [ ]:
try:
    con.sql("""
        SELECT order_id, COUNT(*) AS items
        FROM order_items
        WHERE COUNT(*) > 1
        GROUP BY order_id
    """).df()
except duckdb.Error as e:
    print("DuckDB refused:", str(e).splitlines()[0])

### Cell 6 — How many orders have more than one item, and how many item rows they hold

In [ ]:
row = con.sql("""
    SELECT COUNT(*) AS orders_that_repeat, SUM(items) AS their_item_rows
    FROM (SELECT order_id, COUNT(*) AS items
          FROM order_items
          GROUP BY order_id
          HAVING COUNT(*) > 1)
""").fetchone()                      # the first row of the result; [0] and [1] are its two columns
print("orders that repeat:", row[0], "  item rows they hold:", row[1])

### Cell 7 — One row per order first: the `WITH` query on its own

In [ ]:
con.sql("""
    WITH order_size AS (
        SELECT order_id, COUNT(*) AS n_items
        FROM order_items
        GROUP BY order_id
    )
    SELECT COUNT(*)                 AS rows_in_order_size,
           COUNT(DISTINCT order_id) AS distinct_orders
    FROM order_size
""").df()

### Cell 8 — Then join: orders, and items per order, by customer state

In [ ]:
con.sql("""
    WITH order_size AS (
        SELECT order_id, COUNT(*) AS n_items
        FROM order_items
        GROUP BY order_id
    )
    SELECT c.customer_state, COUNT(*) AS orders,
           ROUND(AVG(s.n_items), 2) AS items_per_order
    FROM order_size AS s
    JOIN orders AS o    ON o.order_id = s.order_id
    JOIN customers AS c ON c.customer_id = o.customer_id
    GROUP BY c.customer_state
    ORDER BY orders DESC LIMIT 5
""").df()

### Cell 9 — The three counts for that join

In [ ]:
con.sql("""
    WITH order_size AS (
        SELECT order_id, COUNT(*) AS n_items
        FROM order_items
        GROUP BY order_id
    )
    SELECT (SELECT COUNT(*) FROM order_size) AS rows_going_in,
           COUNT(*)                          AS rows_coming_out,
           COUNT(DISTINCT s.order_id)        AS distinct_orders_out
    FROM order_size AS s
    JOIN orders AS o    ON o.order_id = s.order_id
    JOIN customers AS c ON c.customer_id = o.customer_id
""").df()

### Cell 10 — The same `WITH`, adding up money: item sales per order, by customer state

In [ ]:
con.sql("""
    WITH order_items_total AS (
        SELECT order_id, SUM(price) AS item_sales
        FROM order_items
        GROUP BY order_id
    )
    SELECT c.customer_state,
           COUNT(*)                    AS orders,
           ROUND(AVG(v.item_sales), 2) AS avg_item_sales
    FROM order_items_total AS v
    JOIN orders AS o    ON o.order_id = v.order_id
    JOIN customers AS c ON c.customer_id = o.customer_id
    GROUP BY c.customer_state
    ORDER BY orders DESC LIMIT 5
""").df()

### Cell 11 — `DISTINCT` patches one column, not the table: São Paulo after the join to its items

In [ ]:
con.sql("""
    SELECT COUNT(*)                   AS rows_in_join,
           COUNT(DISTINCT o.order_id) AS orders,
           ROUND(SUM(i.price), 2)     AS item_sales,
           ROUND(AVG(i.price), 2)     AS avg_price
    FROM orders AS o
    JOIN customers AS c   ON c.customer_id = o.customer_id
    JOIN order_items AS i ON i.order_id = o.order_id
    WHERE c.customer_state = 'SP'
""").df()

## From Block 3: the cells this lab cites

Copied from the Lab 3 project's `lecture.ipynb`, with Block 3's numbering. They run on this project's `orders`, `order_items` and `customers`.

### Block 3, cell 1 — The first join: every order, with its customer's state

In [ ]:
con.sql("""
    SELECT o.order_id, o.order_status, c.customer_state
    FROM orders AS o
    JOIN customers AS c ON c.customer_id = o.customer_id
    ORDER BY o.order_id
    LIMIT 5
""").df()

### Block 3, cell 2 — The check after a join: the rows and the distinct orders that came out

In [ ]:
con.sql("""
    SELECT COUNT(*)                   AS rows_in_result,
           COUNT(DISTINCT o.order_id) AS distinct_orders_in_result
    FROM orders AS o
    JOIN customers AS c ON c.customer_id = o.customer_id
""").df()

### Block 3, cell 3 — Join, then `GROUP BY`: orders per customer state

In [ ]:
con.sql("""
    SELECT c.customer_state, COUNT(*) AS orders
    FROM orders AS o
    JOIN customers AS c ON c.customer_id = o.customer_id
    GROUP BY c.customer_state
    ORDER BY orders DESC
    LIMIT 5
""").df()

### Block 3, cell 9 — The rows with no partner: orders that have no items

In [ ]:
con.sql("""
    SELECT COUNT(*) AS orders_with_no_items
    FROM orders AS o
    LEFT JOIN order_items AS i ON i.order_id = o.order_id
    WHERE i.order_id IS NULL
""").df()

### Block 3, cell 10 — What are they? The orders with no items, by status

In [ ]:
con.sql("""
    SELECT o.order_status, COUNT(*) AS orders
    FROM orders AS o
    LEFT JOIN order_items AS i ON i.order_id = o.order_id
    WHERE i.order_id IS NULL
    GROUP BY o.order_status
    ORDER BY orders DESC
""").df()

## What to remember from Block 4

1. **After `orders JOIN order_items`, one row is one item, not one order.** An order can have several items, so it
   is in the result once per item. That is a fact about this join, not a law of joins: Block 3's `LEFT JOIN` of the
   same two tables also has 43 rows that are an order with no item. Say what one row is, for the join you wrote,
   before you count or average anything.
2. **A join is right or wrong for a question, not by itself.** Joined to `order_items`, São Paulo has 1,278 rows.
   That is the right answer to "how many items" and the wrong answer to "how many orders" (1,133). The query ran
   both times; only the sentence tells them apart.
3. **`COUNT(*)` counts rows, and `AVG` averages rows, whatever the column is called.** Naming a column `orders` does
   not make its rows orders. And the same join had already dropped São Paulo's 17 orders with no items, without a
   word: 1,116 of its 1,133 orders have items.
4. **`HAVING` is a `WHERE` for groups.** `HAVING COUNT(*) > 1` keeps the groups that have more than one row; it
   runs after `GROUP BY`. `WHERE` filters rows before any group exists, so it cannot hold a count (DuckDB refuses:
   *WHERE clause cannot contain aggregates*). On the map of the order a query runs: `FROM`, `WHERE`, `GROUP BY`,
   `HAVING`, `SELECT`, `ORDER BY`, `LIMIT`.
5. **`GROUP BY key HAVING COUNT(*) > 1` finds the keys that repeat.** 265 orders have more than one item, and they
   hold 679 item rows. Their 679 − 265 = 414 extra rows are the whole gap between the join's 3,024 rows and its
   2,610 distinct orders. Before you join to a table, ask which of its keys repeat.
6. **The repair: one row per order first, then join.** `WITH order_size AS (… GROUP BY order_id)` is a table with
   one row per order (2,610 rows). Joined to `orders` and `customers`, every row finds exactly one partner, so
   `COUNT(*)` counts orders again. The three counts prove it: 2,610 · 2,610 · 2,610.
7. **The rule.** Start from the table whose rows are what the question counts. If the number you need lives on a
   table with several rows per order, add it up to one row per order first, in a `WITH`, and then join.
8. **`DISTINCT` is a patch, one column at a time.** São Paulo after the join to its items: `SUM(i.price)` is
   143,900.82, and it was never wrong, because the price lives on the item. `COUNT(*)` is 1,278, wrong for orders,
   and `COUNT(DISTINCT o.order_id)` counts the orders with items: 1,116. `AVG(i.price)` is 112.60, an average per item; the average per order is 128.94 (cell 10),
   which `AVG` cannot give here and money over orders, `SUM(i.price) / COUNT(DISTINCT o.order_id)`, can. If the question is about orders, fix the rows,
   not the column.
9. **The three counts go with every join.** In the lab and the homework, the check cells hold them.

## Common mistakes

- **Counting rows and calling them orders.** After a join to a table with several rows per order, `COUNT(*)` counts the rows of the
  result: items, if you joined `order_items`. Say what one row of the result is before you count it.
- **Averaging rows and calling it "per order".** `AVG` after the same join is an average over items. `AVG` for
  an average per order needs one row per order.
- **A condition on a count in `WHERE`.** `WHERE COUNT(*) > 1` is refused: `WHERE` sees one row at a time, before
  any group exists. A condition on a group goes in `HAVING`, after `GROUP BY`.
- **Patching with `DISTINCT`.** `COUNT(DISTINCT o.order_id)` repairs that one count. An `AVG` in the same query is
  still an average over items, not over orders: changing the count does not change it. Money over orders,
  `SUM(…) / COUNT(DISTINCT …)`, is a second patch; one row per order first fixes every column at once.
- **Trusting a total because it looks plausible.** São Paulo's 1,278 was 17 orders lost and 162 rows too many at
  once. Run the three counts, not only the total.
- **Carrying a parent's number down to its children.** A number that belongs to an order, joined to that order's
  items, is repeated once per item, and a `SUM` of it counts a three-item order three times. Group the items into one
  row per order first, then join them to the order's number.
- **Forgetting who the `WITH` left out.** `order_size` has a row only for orders that have items: 2,610 of 2,653.
  Whether the 43 others belong in the answer is a question about the sentence; Block 3's `LEFT JOIN … IS NULL`
  finds them.